In [ ]:
# # Install scikit-rf if not already available
# %pip install scikit-rf matplotlib

In [ ]:
import skrf as rf
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import shutil
import re
from datetime import datetime

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _run_suffix(path: Path) -> str:
    """Extract run token e.g. '-0001' from a stem, even when followed by '-parameter'.

    Matches the last -NNN... that is followed by either a non-digit suffix or end-of-stem.
      MWS-run-0001.s2p                 → '-0001'
      MWS-run-0001-parameter.txt       → '-0001'
      Sim1_field_lam_12_MWS-sweep-01-0003.s2p → '-0003'
    """
    m = re.search(r'(-\d+)(?=-\D|$)', path.stem)
    return m.group(1) if m else ''

def _name_contains(name: str, token: str) -> bool:
    """Case-insensitive substring test — is `token` present in `name`?

    Config-agnostic: the caller supplies whatever token(s) it cares about
    (e.g. 'Gridded', 'Solid', 'Mesh', ...) and loops over them itself.
    """
    return token.lower() in name.lower()

def _read_pressure(s2p_path: Path) -> float | None:
    """Pressure [Torr] for an s2p run.

    1) Matching '*-parameter.txt' (same run suffix) containing 'Pressure=<num>'.
    2) Fallback: a 'NNNTorr' token in the folder/file path (e.g. '..._500Torr').
    """
    suffix = _run_suffix(s2p_path)
    for txt in s2p_path.parent.glob('*.txt'):
        if _run_suffix(txt) == suffix:
            m = re.search(r'Pressure\s*=\s*([+-]?[\d.]+(?:[eE][+-]?\d+)?)',
                          txt.read_text(errors='ignore'))
            if m:
                return float(m.group(1))
            break  # matched the run's txt but it had no Pressure= line
    m = re.search(r'(\d+(?:\.\d+)?)\s*[Tt]orr', str(s2p_path))
    return float(m.group(1)) if m else None


In [ ]:
from collections import defaultdict

# Root holding one folder per CST simulation, each with a 'TOUCHSTONE files' subfolder.
ROOT    = Path('/home/vpodolsky/CST Studio Result Files')
# Config tokens to look for in each simulation's folder name. Extend as needed;
# folders matching none of these are skipped.
CONFIGS = ['Gridded', 'Solid']

# --- Discover every .s2p, tag it with config + pressure (no copy/rename/delete) ---
records = []
for s2p in sorted(ROOT.glob('*/TOUCHSTONE files/*.s2p')):
    sim_folder = s2p.parents[1].name          # e.g. 'Scooter_Curved_Anode_Solid_Full_0Torr'
    config     = next((c for c in CONFIGS if _name_contains(sim_folder, c)), None)
    if config is None:                         # folder matches no known config token
        continue
    records.append({
        'config'    : config,
        'pressure'  : _read_pressure(s2p),
        'sim_folder': sim_folder,
        'path'      : s2p,
        'network'   : rf.Network(str(s2p)),
    })

# --- Warn on duplicate (config, pressure) pairs (grouped keeps only the last) ---
seen = defaultdict(list)
for r in records:
    seen[(r['config'], r['pressure'])].append(r['path'].name)
dupes = {k: v for k, v in seen.items() if len(v) > 1}
if dupes:
    print('⚠️  Duplicate (config, pressure) run(s) — grouped[] keeps only the last; all remain in records:')
    for (cfg, p), names in dupes.items():
        print(f'    {cfg} @ {p} Torr: {names}')
    print()

# --- Nested view: config -> pressure -> Network (sorted by pressure) ---
grouped = defaultdict(dict)
for r in records:
    grouped[r['config']][r['pressure']] = r['network']
grouped = {cfg: dict(sorted(d.items(), key=lambda kv: (kv[0] is None, kv[0])))
           for cfg, d in grouped.items()}

# --- Report ---
print(f'Indexed {len(records)} run(s) across configs: {sorted(grouped)}\n')
for cfg, d in grouped.items():
    print(f'{cfg}: {len(d)} run(s) at {sorted(p for p in d if p is not None)} Torr')
print()
for r in records:
    print(f"  {r['config']:<8} {str(r['pressure']) + ' Torr':<12} {r['sim_folder']}/…/{r['path'].name}")


In [ ]:
# --- Pick one configuration to drive the plotting cells below ---
active_config = 'Gridded'          # 'Solid' or 'Gridded'
run_name      = active_config

sel             = grouped[active_config]            # {pressure: Network}
swept_pressures = sorted(p for p in sel)            # ascending pressures
networks        = [sel[p] for p in swept_pressures]
s2p_files       = [next(r['path'] for r in records
                        if r['config'] == active_config and r['pressure'] == p)
                   for p in swept_pressures]

baseline_idx = next((i for i, p in enumerate(swept_pressures) if p == 0), 0)
pressures    = [p for p in swept_pressures if p not in (None, 0)]

print(f'Active config: {active_config}  ({len(networks)} run(s))')
for p, f in zip(swept_pressures, s2p_files):
    print(f'  {str(p) + " Torr":<12} {f.name}')
print(f'\nbaseline_idx = {baseline_idx}  (pressure = {swept_pressures[baseline_idx]} Torr)')
print(f'Non-zero pressures: {pressures}')


In [ ]:
networks = [rf.Network(str(f)) for f in s2p_files]
ntwk = networks[0]
print(f'Network: {ntwk.name}')
print(f'Frequency range: {ntwk.f[0]/1e9:.3f} – {ntwk.f[-1]/1e9:.3f} GHz  ({len(ntwk.f)} points)')
print(f'Number of ports: {ntwk.nports}')

In [ ]:
# S11 → index (0,0), S21 → index (1,0)
params = {'S11': (0, 0), 'S21': (1, 0)}

cmap=plt.cm.tab10
colors=[cmap(i) for i,_ in enumerate(networks)]
for idx, ntwk in enumerate(networks):
    freq_ghz = ntwk.f / 1e9

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f'{run_name}, S-Parameters — {ntwk.name} at {swept_pressures[idx]} Torr', fontsize=13)

    # --- Magnitude (dB): S11 and S21 overlaid ---
    ax = axes[0]
    for label, (i, j) in params.items():
        mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
        ax.plot(freq_ghz, mag_db, label=label)
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(f'{run_name} Magnitude')
    ax.legend()
    ax.grid(True, alpha=0.4)

    # --- Phase: S21 ---
    ax = axes[1]
    i, j = params['S21']
    ax.plot(freq_ghz, np.angle(ntwk.s[:, i, j], deg=True), color='tab:orange')
    ax.set_xlim([min(freq_ghz),max(freq_ghz)])
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Phase (degrees)')
    ax.set_title(f'{run_name} S21 Phase')
    ax.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

# --- S21 magnitude zoom: all pressures overlaid ---
fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.set_title(f'{run_name}, S21 Magnitude — all pressures')
i, j = params['S21']
for idx, ntwk in enumerate(networks):
    mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
    ax2.plot(ntwk.f / 1e9, mag_db, color=colors[idx], label=f'{swept_pressures[idx]} Torr')
ax2.set_xlabel('Frequency (GHz)')
ax2.set_ylabel('S21 (dB)')
ax2.legend()
ax2.grid(True, alpha=0.4)
# ax2.set_ylim(-1, 1)
ax2.set_xlim([min(freq_ghz),max(freq_ghz)])
plt.tight_layout()
plt.show()

# --- S11 and S21 all pressures on one plot ---
fig3, ax3 = plt.subplots(figsize=(10, 5))
ax3.set_title(f'{run_name}, S11 & S21 Magnitude — all pressures')
for idx, ntwk in enumerate(networks):
    freq = ntwk.f / 1e9
    for label, (i, j) in params.items():
        mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
        ls = '-' if label == 'S21' else ':'
        ax3.plot(freq, mag_db, color=colors[idx], linestyle=ls,
                 label=f'{label} {swept_pressures[idx]} Torr')
ax3.set_xlabel('Frequency (GHz)')
ax3.set_ylabel('Magnitude (dB)')
ax3.set_xlim([min(freq_ghz),max(freq_ghz)])
ax3.legend(ncol=2, fontsize=8)
ax3.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# --- S11 and S21 all pressures on one plot ---
fig3, ax3 = plt.subplots(figsize=(10, 5))
ax3.set_title(f'{run_name}, S11 & S21 Magnitude — all pressures')
for idx, ntwk in enumerate(networks):
    freq = ntwk.f / 1e9
    for label, (i, j) in params.items():
        mag_db = 20 * np.log10(np.abs(ntwk.s[:, i, j]) + 1e-30)
        ls = '-' if label == 'S21' else ':'
        ax3.plot(freq, mag_db, color=colors[idx], linestyle=ls,
                 label=f'{label} {swept_pressures[idx]} Torr')
ax3.set_xlabel('Frequency (GHz)')
ax3.set_ylabel('Magnitude (dB)')
# ax3.set_xlim([min(freq_ghz),max(freq_ghz)])
ax3.legend(ncol=2, fontsize=8)
ax3.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# --- Phase: S21 ---

fig,axes=plt.subplots(3,1,figsize=(10,18))
i, j = params['S21']
cmap=plt.cm.tab10
colors=[cmap(i) for i,_ in enumerate(networks)]
phase_0=np.unwrap(np.angle(networks[0].s[:, i, j]))*180/np.pi

for ii,ntwk in enumerate(networks):
    ax=axes[0]
    ax.plot(freq_ghz, np.angle(ntwk.s[:, i, j]), color=colors[ii])
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('Phase (rads)')
    ax.set_title(f'{run_name}, S21 Phase')
    ax.grid(True, alpha=0.4)

    # ax = axes[1]
    # ax.plot(freq_ghz, np.unwrap(np.angle(ntwk.s[:, i, j])), color=colors[ii])
    # ax.set_xlabel('Frequency (GHz)')
    # ax.set_ylabel('Unwrapped Phase (Rad)')
    # ax.set_title(f'{run_name}, S21 Phase')
    # ax.grid(True, alpha=0.4)

    ax = axes[1]
    ax.plot(freq_ghz, np.unwrap(np.angle(ntwk.s[:, i, j]))*180/np.pi, color=colors[ii])
    ax.set_xlabel('Frequency [GHz]')
    ax.set_ylabel('Unwrapped Phase [deg]')
    ax.set_title(f'{run_name}, S21 Phase')
    # ax.set_xlim([59,59.3])
    ax.set_ylim([-4500,500])

    ax = axes[2]
    phase_n=np.unwrap(np.angle(ntwk.s[:, i, j]))*180/np.pi
    ax.plot(freq_ghz, phase_n-phase_0, color=colors[ii])
    ax.set_xlabel('Frequency [GHz]')
    ax.set_ylabel('dif Unwrapped Phase [deg]')
    ax.set_title(f'{run_name}, S21 Phase')
    fig.legend([f'{p} Torr' for p in swept_pressures],loc=(0.81,0.90))
    # ax.set_xlim([59,59.3])
    # ax.set_ylim([-100,0])

    ax.grid(True, alpha=0.4)

In [ ]:
def get_analytic_permitivity(p, T):
    """Return (permittivity, refractive_index) of N2 at pressure p [Torr] and temperature T [K]."""
    n_N2_STP = 1.0002976          # N2 refractive index at 760 Torr, 0 °C
    n = 1 + (n_N2_STP - 1) * (p / 760) * (273.15 / T)
    return n**2, n

pressures = swept_pressures
# pressures=[0.0, 1.0, 10.0]
# colors = ['tab:blue', 'tab:orange', 'tab:green']
n_vac=1


T = 293.15          # K
c = 2.99792458e8    # m/s --speed of light
d = 0.226          # m  — path length

f_hz     = networks[0].f
lam=c/f_hz #m--MWI wavelength
freq_ghz = f_hz / 1e9

# baseline_idx    = 0
i, j            = params['S21']
phase_unwrapped = [np.unwrap(np.angle(ntwk.s[:, i, j])) * 180 / np.pi
                   for ntwk in networks]
print(baseline_idx)
baseline_phase  = phase_unwrapped[baseline_idx]

dphi_sim      = []
dphi_rad_p_T = []

# --- Phase difference: simulation vs analytic ---
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_title(rf'{run_name}, $\angle S21(p) - \angle S21_{{vacuum}}$')

for idx, (ntwk, phase) in enumerate(zip(networks, phase_unwrapped)):
    if idx == baseline_idx:
        continue
    dphi = phase - baseline_phase
    dphi_sim.append(dphi)
    ax.plot(freq_ghz, dphi, label=f'{ntwk.name} ({pressures[idx]} Torr)',color=colors[idx])

for k,p_torr in enumerate(pressures):
    if k == baseline_idx:
        continue
    eps, n_N2_P = get_analytic_permitivity(p_torr, T)
    dphi_a=-(n_N2_P-n_vac)*d*2*np.pi/lam*180/np.pi
    dphi_rad_p_T.append(dphi_a)
    ax.plot(freq_ghz, dphi_a, linestyle='--', color=colors[k])#label=f'{p_torr} Torr analytic',

ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('ΔPhase (degrees)')
# ax.set_xlim([40,67])
ax.set_ylim([-150,150])
ax.legend(loc='lower center',ncol=2)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# --- Residual: analytic − simulation ---
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.set_title(rf'{run_name}, $(Δ\Phi_{{sim}}-Δ\Phi_{{analytic}})/Δ\Phi_{{analytic}}$')
for k, p_torr in enumerate(pressures):
    if k == baseline_idx:
        continue
    ax2.plot(freq_ghz, (dphi_sim[k-1]-dphi_rad_p_T[k-1])/dphi_rad_p_T[k-1] *100, label=f'{p_torr} Torr', color=colors[k])
ax2.set_xlabel('Frequency (GHz)')
ax2.set_ylabel(' Percent Error (%)')
ax2.set_ylim([-500,500])
# ax2.set_xlim([40,67])
# ax2.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax2.legend(loc='lower center',ncol=2)
ax2.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
freq_to_plot = np.linspace(59, 67, 9)  # GHz
plot_colors = plt.cm.tab10(np.linspace(0, 1, len(freq_to_plot)))

non_baseline_pressures = [pressures[idx] for idx in range(len(networks)) if idx != baseline_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(f'{run_name}, ΔPhase vs Pressure')
print(len(freq_ghz))
for color, f_target in zip(plot_colors, freq_to_plot):
    freq_idx = np.argmin(np.abs(freq_ghz - f_target))
    dphi_at_f = [dphi_sim[k][freq_idx] for k in range(len(dphi_sim))]
    dphi_a_at_f = [dphi_rad_p_T[k][freq_idx] for k in range(len(dphi_rad_p_T))]
    ax.plot(non_baseline_pressures, dphi_at_f, color=color, marker='o',
            label=f'{freq_ghz[freq_idx]:.1f} GHz')
    ax.plot(non_baseline_pressures, dphi_a_at_f, color=color, linestyle='--')

ax.set_xlabel('Pressure (Torr)')
ax.set_ylabel('ΔPhase (degrees)')
ax.legend(title='Frequency')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()